# Lab ml_01 — Linear & logistic regression from scratch

**Goals**
1. Linear regression two ways: closed-form **normal equations** and **gradient descent** — and check they agree.
2. Logistic regression trained with the **binary cross-entropy**, using the gradient you derived on paper,
   $\nabla_w L = \tfrac1n X^\top(\sigma(Xw) - y)$.
3. Verify the gradient with **finite differences**, then plot the **decision boundary**.

Do the paper homework for **Linear regression** (and ideally **Logistic regression**) first.

**Reading**: Stanford CS229 main notes ch. 1–2 (https://cs229.stanford.edu/main_notes.pdf);
MIT 6.390 notes: Regression, Gradient Descent, Classification (https://introml.mit.edu/notes/).

## Theory recap

* Least squares: minimise $\tfrac1n\|Xw - y\|^2$. Setting the gradient $\tfrac2n X^\top(Xw-y)$ to zero gives the
  **normal equations** $X^\top X\,w = X^\top y$ (solve them with `np.linalg.solve`, never `inv`).
* Gradient descent: $w \leftarrow w - \eta\,\tfrac2n X^\top(Xw - y)$.
* Logistic model: $p = \sigma(z)$, $z = Xw$, $\sigma(z) = 1/(1+e^{-z})$. Mean BCE:
  $L(w) = -\tfrac1n\sum_i [y_i\ln p_i + (1-y_i)\ln(1-p_i)]$, and $\partial L/\partial z_i = (p_i - y_i)/n$.
* Stable forms: $-\ln\sigma(z) = \log(1+e^{-z})$; use `np.logaddexp(0, z)`; compute $\sigma$ without overflowing `exp`.
* Central differences: $\partial_i f(w) \approx \dfrac{f(w + \varepsilon e_i) - f(w - \varepsilon e_i)}{2\varepsilon}$.

Convention in this lab: a bias is handled by appending a column of ones to $X$ (`add_bias`), so the last weight is the intercept.

In [ ]:
import sys, math, re, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# make `lc_client` importable whether Jupyter was started in labs/ or in the repo root
LAB_DIR = Path.cwd() if (Path.cwd() / "lc_client.py").exists() else Path.cwd() / "labs"
sys.path.insert(0, str(LAB_DIR))
DATA = LAB_DIR / "data"

def add_bias(X):
    """Append a column of ones so the last weight is the intercept. Provided."""
    X = np.asarray(X, dtype=float)
    return np.hstack([X, np.ones((X.shape[0], 1))])

rng = np.random.default_rng(0)
x1 = rng.uniform(-3, 3, 50)
y1 = 1.7 * x1 - 0.5 + rng.normal(0, 0.8, 50)
X1 = add_bias(x1[:, None])
plt.scatter(x1, y1, s=12); plt.title("synthetic 1-D regression data"); plt.show()

## 1. Linear regression

In [ ]:
def fit_normal_equations(X, y):
    """Least-squares weights solving (X^T X) w = X^T y."""
    # TODO
    raise NotImplementedError


def mse_loss(w, X, y):
    # TODO: mean of squared residuals
    raise NotImplementedError


def mse_grad(w, X, y):
    # TODO: 2/n X^T (Xw - y)
    raise NotImplementedError


def fit_gd(X, y, lr=0.1, steps=2000):
    """Batch gradient descent on the MSE from w = 0; return the final w."""
    # TODO
    raise NotImplementedError

In [ ]:
w_ne = fit_normal_equations(X1, y1)
w_gd = fit_gd(X1, y1, lr=0.05, steps=2000)
print("normal equations:", w_ne, "  gradient descent:", w_gd)
xs = np.linspace(-3, 3, 100)
plt.scatter(x1, y1, s=12)
plt.plot(xs, add_bias(xs[:, None]) @ w_ne, "r", label="fit")
plt.legend(); plt.show()

# Try: what happens to GD with lr = 0.5? Plot mse_loss over the iterations for a few learning rates.

## 2. Logistic regression

In [ ]:
def sigmoid(z):
    """Numerically stable logistic function (no overflow warnings for |z| ~ 1000)."""
    # TODO: treat z >= 0 and z < 0 separately (or use np.exp only on non-positive arguments)
    raise NotImplementedError


def bce_loss(w, X, y):
    """Mean binary cross-entropy (nats) of sigmoid(Xw) against labels y in {0,1}."""
    # TODO: stable form: mean(logaddexp(0, z) - y*z)
    raise NotImplementedError


def bce_grad(w, X, y):
    """Your hand-derived gradient X^T (sigmoid(Xw) - y) / n."""
    # TODO
    raise NotImplementedError


def numerical_grad(f, w, eps=1e-6):
    """Central finite differences of a scalar function f at w."""
    # TODO
    raise NotImplementedError


def fit_logistic(X, y, lr=0.5, steps=3000):
    """Gradient descent on the mean BCE from w = 0. Return (w, loss_history)."""
    # TODO: record bce_loss before every step
    raise NotImplementedError


def accuracy(w, X, y):
    return float(np.mean((sigmoid(X @ w) >= 0.5) == (y == 1)))

In [ ]:
rng = np.random.default_rng(1)
A = rng.normal([-1.5, -1.0], 0.9, size=(100, 2)); B = rng.normal([1.5, 1.0], 0.9, size=(100, 2))
X2 = add_bias(np.vstack([A, B])); y2 = np.r_[np.zeros(100), np.ones(100)]

w0 = rng.normal(size=3)
print("analytic :", bce_grad(w0, X2, y2))
print("numerical:", numerical_grad(lambda w: bce_loss(w, X2, y2), w0))

w_lr, hist = fit_logistic(X2, y2)
print("weights", w_lr, " accuracy", accuracy(w_lr, X2, y2))
plt.plot(hist); plt.xlabel("step"); plt.ylabel("mean BCE"); plt.yscale("log"); plt.show()

## 3. Decision boundary

The boundary is $w_1x_1 + w_2x_2 + b = 0$. Plot it over the data together with a contour of $P(y=1\mid x)$.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
P = sigmoid(add_bias(np.c_[g1.ravel(), g2.ravel()]) @ w_lr).reshape(g1.shape)
plt.contourf(g1, g2, P, levels=20, cmap="RdBu_r", alpha=0.6)
plt.contour(g1, g2, P, levels=[0.5], colors="k")
plt.scatter(A[:, 0], A[:, 1], s=10, c="tab:blue"); plt.scatter(B[:, 0], B[:, 1], s=10, c="tab:red")
plt.title("P(y=1|x) and the decision boundary"); plt.show()

*Your observations: what happens to $\|w\|$ if you make the classes perfectly separable and train for longer?*

In [ ]:
from lc_client import grade
grade("ml_01_regression_from_scratch", globals())